<a href="https://colab.research.google.com/github/NicoGiba09/MyV7/blob/main/Maquina-v7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ☁️ Maquina-v7 — Cloud PC Linux + LXQt + Apache Guacamole en Google Colab

> Escritorio remoto **GNU/Linux (LXQt)** accesible desde el **navegador** mediante **Apache Guacamole**.
> Sin necesidad de instalar clientes RDP/VNC. Funciona en cualquier navegador web.

**Arquitectura:**
```
Google Colab → Linux → Xvfb → LXQt → x11vnc → guacd → Guacamole → Browser
```

**Pasos:** ejecuta las celdas en orden: 1) Configuracion, 2) Instalar, 3) Iniciar, 4) Estado.

In [9]:
#@title ⚙️ Configuración
GITHUB_USER = "jephersonRD"   #@param {type:"string"}
REPO        = "Maquina-v7"    #@param {type:"string"}

USERNAME    = "admin"         #@param {type:"string"}
PASSWORD    = "applejr.net"         #@param {type:"string"}
RESOLUTION  = "1280x720"      #@param {type:"string"}
GUAC_PORT   = 8080            #@param {type:"integer"}

print("✅ Configuración lista.")
print(f"   Resolución: {RESOLUTION}")
print(f"   Puerto Guacamole: {GUAC_PORT}")
print(f"   Usuario: {USERNAME}")

✅ Configuración lista.
   Resolución: 1280x720
   Puerto Guacamole: 8080
   Usuario: admin


In [10]:
#@title 📦 Instalar LXQt + Guacamole + Audio (Modo Estrito com Parada em Erros)
import os
import sys

def run_command(cmd, description):
    print("\n" + "="*50)
    print(f"{description}...")
    print("="*50)

    # Executa o comando e captura o status de saída
    ret = !{cmd}

    # Se o comando retornar erro, interrompe a execução do Python
    if _exit_code != 0:
        print(f"\n❌ ERRO FATAL: Falha ao executar '{description}'.")
        print(f"Código de erro: {_exit_code}")
        sys.exit(1)

repo_dir = "/content/Maquina-v7"

# 1. Download do repositório com validação
if not os.path.isdir(repo_dir):
    print("📥 Descargando repositorio...")
    url = f"https://codeload.github.com/{GITHUB_USER}/{REPO}/tar.gz/refs/heads/main"

    !curl -fsSL "{url}" -o /tmp/maquina-v7.tar.gz

    if not os.path.exists("/tmp/maquina-v7.tar.gz") or os.path.getsize("/tmp/maquina-v7.tar.gz") == 0:
        print("❌ ERRO FATAL: O repositório não pôde ser baixado. Verifique se o usuário/repositório existem e são públicos.")
        sys.exit(1)

    !tar xzf /tmp/maquina-v7.tar.gz -C /content
    !mv /content/{REPO}-main /content/Maquina-v7 2>/dev/null || !mv /content/{REPO}-master /content/Maquina-v7
    !rm -f /tmp/maquina-v7.tar.gz

# Entra na pasta dos scripts
%cd /content/Maquina-v7/scripts
!chmod +x *.sh

# 2. Execução das etapas com interrupção em caso de erro (set -e obriga o bash a parar)
run_command(f"bash -e setup_lxqt.sh {RESOLUTION}", "📦 [1/4] Instalando LXQt")
run_command("bash -e setup_audio.sh", "🔊 [2/4] Configurando audio")
run_command("bash -e setup_vnc.sh", "🔗 [3/4] Instalando VNC")
run_command("bash -e setup_guacamole.sh", "🌐 [4/4] Instalando Apache Guacamole")

print("\n✅ Instalación completada con éxito.")

/content/Maquina-v7/scripts

📦 [1/4] Instalando LXQt...

🔊 [2/4] Configurando audio...

🔗 [3/4] Instalando VNC...

🌐 [4/4] Instalando Apache Guacamole...

✅ Instalación completada con éxito.


In [ ]:
#@title 🚀 Iniciar Escritorio (Con espera inteligente de URL)
import os
import re
import time
import sys

# Garante que está na pasta correta
%cd /content/Maquina-v7/scripts

print("🔄 Iniciando servicios de escritorio...")
!chmod +x start_desktop.sh
!bash start_desktop.sh {RESOLUTION} {USERNAME} {PASSWORD} {GUAC_PORT}

# Aguarda a inicialização do Cloudflare e busca a URL pública no arquivo de log
print("\n⏳ Esperando a que Cloudflare Tunnel genere la URL pública (máx. 30s)...")
public_url = ""
log_file = "/tmp/cloudflared.log"

for attempt in range(30):
    if os.path.exists(log_file):
        try:
            with open(log_file, "r") as f:
                content = f.read()
                match = re.search(r'https://[a-zA-Z0-9._-]+\.trycloudflare\.com', content)
                if match:
                    public_url = match.group(0)
                    break
        except Exception:
            pass
    time.sleep(1)

print("\n" + "="*50)
print("  🌐 COMO ACCEDER:")
print("="*50)

if public_url:
    print(f"  ✅ URL PÚBLICA (accesible desde cualquier dispositivo):")
    print(f"     👉 {public_url}")
else:
    print("  ⚠️ No se detectó la URL pública automáticamente.")
    print("  Verifica si el proceso 'cloudflared' se está ejecutando o revisa el log manualmente con:")
    print("  !cat /tmp/cloudflared.log")

print(f"\n  📍 URL local (solo dentro de Colab): http://localhost:{GUAC_PORT}")
print(f"  👤 Usuario : {USERNAME}")
print(f"  🔑 Password: {PASSWORD}")
print("="*50)

/content/Maquina-v7/scripts
🔄 Iniciando servicios de escritorio...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🚀 MAQUINA-V7 — Iniciando escritorio
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  🖥️  Desktop      : LXQt
  🌐 Gateway      : Apache Guacamole
  🔗 Backend      : VNC (x11vnc)
  📺 Resolucion   : 1280x720

  🎮 Detectando GPU...
  🎮 GPU          : No detectada (CPU)

   ↳ Deteniendo procesos previos...
   ↳ [1/6] Iniciando Xvfb (display :10)...
      ✅ Xvfb listo
   ↳ [2/6] Iniciando PulseAudio...
      ✅ PulseAudio listo
      🔊 Audio Monitor: auto_null.monitor
   ↳ [3/6] Iniciando LXQt...
"Icon Theme not set. Fallbacking to Oxygen, if installed"
"Fallback Icon Theme (Oxygen) not found"
lxqt-session: LockScreenManager couldn't start
      ✅ LXQt listo (PID: 84370)
   ↳ [4/6] Iniciando x11vnc...
      ✅ x11vnc listo (PID: 84391)
   ↳ [5/6] Iniciando guacd...
      ✅ guacd listo (PID: 84401)
   ↳ [6/6] Iniciando Apache Guacamole...
start_desktop.sh: line 160: /etc/guacamol

In [4]:
#@title 📊 Estado del Sistema
%cd /content/Maquina-v7/scripts
!bash diagnostics.sh

/content/Maquina-v7/scripts
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🔍 MAQUINA-V7 — Diagnostico del sistema
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

【1. Sistema】
   OS: Ubuntu 24.04.4 LTS
   Kernel: 6.6.122+
   Arquitectura: x86_64

【2. GPU】
   ⚠️  No se detecto GPU NVIDIA

【3. Display (X11)】
diagnostics.sh: line 32: DISPLAY: unbound variable


In [ ]:
#@title 🔒 Keep-Alive (anti-apagado de Colab)
import threading, time

def _heartbeat():
    while True:
        time.sleep(60)
        print("♥ keep-alive", time.strftime('%H:%M:%S'))
threading.Thread(target=_heartbeat, daemon=True).start()
print('✅ Keep-alive activado.')
print('⚠️ NO cierres ni ocultes esta pestaña de Colab.')

In [ ]:
#@title 🎮 Instalar Steam (opcional)
INSTALL_STEAM = False  #@param {type:"boolean"}
if INSTALL_STEAM:
    %cd /content/Maquina-v7/scripts
    !bash install_steam.sh
else:
    print('Omitido (INSTALL_STEAM = False).')